**Công việc:**  
Đọc nhãn cụm K = 4 đã lưu.

Lấy 41 Work Activities LV từ SQLite.

Tính mức độ đặc trưng của từng hoạt động trong mỗi cụm.

Tìm 5 nghề gần tâm cụm nhất để làm ví dụ tiêu biểu.

Xuất báo cáo CSV để bạn đọc và đặt tên cụm.

In [1]:

from pathlib import Path
import sqlite3

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

In [2]:
# =========================
# 1. Cấu hình
# =========================
DB_PATH = Path("../data/database/career.db")
ASSIGNMENT_PATH = Path(
    "../outputs/kmeans_experiments/work_activities_41_lv_k4.csv"
)
OUTPUT_DIR = Path("../outputs/cluster_analysis")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
assignments = pd.read_csv(ASSIGNMENT_PATH)

# In tên cột để kiểm tra cấu trúc file
print("Assignment columns:", assignments.columns.tolist())

Assignment columns: ['occupation_code', 'cluster_id']


In [4]:
cluster_col = 'cluster_id'

if "occupation_code" not in assignments.columns:
    raise ValueError("File assignment thiếu occupation_code")

assignments = assignments[["occupation_code", cluster_col]].copy()
assignments = assignments.rename(columns={cluster_col: "cluster"})

In [6]:
# =========================
# 3. Đọc Work Activities LV
# =========================
with sqlite3.connect(DB_PATH) as conn:
    activities = pd.read_sql_query(
        """
        SELECT occupation_code, activity, level_score
        FROM occupation_work_activities
        """,
        conn
    )

    occupations = pd.read_sql_query(
        """
        SELECT occupation_code, occupation_title
        FROM occupations
        """,
        conn
    )



In [7]:
# Chuyển dữ liệu dạng dài thành 41 cột hoạt động
X = activities.pivot(
    index="occupation_code",
    columns="activity",
    values="level_score"
)

# Chỉ giữ các nghề có đủ dữ liệu hoạt động
X = X.dropna()
before_count = len(X)
# Chỉ giữ nghề có cả dữ liệu LV và nhãn cụm
common_codes = X.index.intersection(
    assignments["occupation_code"]
)
after_count = len(common_codes)


In [8]:
print(f"Số dòng trong X trước intersection: {before_count}")
print(f"Số occupation_code chung: {after_count}")
print(f"Số dòng bị loại: {before_count - after_count}")

Số dòng trong X trước intersection: 911
Số occupation_code chung: 911
Số dòng bị loại: 0


In [13]:
print(type(common_codes))
print(common_codes)

<class 'pandas.Index'>
Index(['11-1011.00', '11-1011.03', '11-1021.00', '11-2011.00', '11-2021.00',
       '11-2022.00', '11-2032.00', '11-2033.00', '11-3012.00', '11-3013.00',
       ...
       '53-7062.00', '53-7062.04', '53-7063.00', '53-7064.00', '53-7065.00',
       '53-7071.00', '53-7072.00', '53-7073.00', '53-7081.00', '53-7121.00'],
      dtype='str', name='occupation_code', length=911)


In [ ]:
X = X.loc[common_codes].sort_index()
 # Chỉ giữ lại những occupation có trong common_codes

In [10]:
labels = (
    assignments
    .drop_duplicates("occupation_code")
    .set_index("occupation_code")
    .loc[common_codes, "cluster"]
)

print("\nSố nghề được phân tích:", len(X))
print("Số hoạt động:", X.shape[1])
print("Số cụm:", labels.nunique())
print("\nSố nghề mỗi cụm:")
print(labels.value_counts().sort_index())



Số nghề được phân tích: 911
Số hoạt động: 41
Số cụm: 4

Số nghề mỗi cụm:
cluster
0    204
1    221
2    258
3    228
Name: count, dtype: int64


In [ ]:
# =========================
# 4. Hồ sơ hoạt động của từng cụm
# =========================
profile = X.assign(cluster=labels).groupby("cluster").mean()
profile.to_csv(
    OUTPUT_DIR / "cluster_activity_profile.csv",
    encoding="utf-8-sig"
)

In [15]:
# Mỗi cụm: 10 hoạt động có Level trung bình cao nhất
top_activities = []

for cluster_id, row in profile.iterrows():
    top = row.sort_values(ascending=False).head(10)

    for rank, (activity, mean_level) in enumerate(top.items(), start=1):
        top_activities.append({
            "cluster": cluster_id,
            "rank": rank,
            "activity": activity,
            "mean_level": mean_level
        })

pd.DataFrame(top_activities).to_csv(
    OUTPUT_DIR / "cluster_top_activities.csv",
    index=False,
    encoding="utf-8-sig"
)

In [16]:
# =========================
# 5. Tìm nghề gần tâm cụm nhất
# =========================
# Chuẩn hóa để các hoạt động có thang đo khác nhau
scaler = StandardScaler()
X_scaled = pd.DataFrame(
    scaler.fit_transform(X),
    index=X.index,
    columns=X.columns
)

representatives = []

for cluster_id in sorted(labels.unique()):
    cluster_codes = labels[labels == cluster_id].index
    cluster_data = X_scaled.loc[cluster_codes]

    # Tâm cụm = vector trung bình các nghề trong cụm
    centroid = cluster_data.mean(axis=0)

    # Khoảng cách Euclidean từ mỗi nghề đến tâm cụm
    distances = np.sqrt(
        ((cluster_data - centroid) ** 2).sum(axis=1)
    )

    nearest_codes = distances.nsmallest(5).index

    for code in nearest_codes:
        representatives.append({
            "cluster": cluster_id,
            "occupation_code": code,
            "title": None,
            "distance_to_centroid": distances.loc[code]
        })

representatives = pd.DataFrame(representatives)

representatives = representatives.drop(columns="title").merge(
    occupations,
    on="occupation_code",
    how="left"
)

representatives.to_csv(
    OUTPUT_DIR / "cluster_representative_occupations.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nĐã xuất báo cáo tại:", OUTPUT_DIR)
print("- cluster_activity_profile.csv")
print("- cluster_top_activities.csv")
print("- cluster_representative_occupations.csv")


Đã xuất báo cáo tại: ..\outputs\cluster_analysis
- cluster_activity_profile.csv
- cluster_top_activities.csv
- cluster_representative_occupations.csv


In [17]:

# =========================
# 6. Tìm hoạt động phân biệt từng cụm
# =========================

overall_mean = X.mean()

cluster_differences = []

for cluster_id in sorted(labels.unique()):
    cluster_codes = labels[labels == cluster_id].index
    other_codes = labels[labels != cluster_id].index

    cluster_mean = X.loc[cluster_codes].mean()
    other_mean = X.loc[other_codes].mean()

    for activity in X.columns:
        cluster_differences.append({
            "cluster": cluster_id,
            "activity": activity,
            "cluster_mean": cluster_mean[activity],
            "overall_mean": overall_mean[activity],
            "other_clusters_mean": other_mean[activity],
            "difference_from_overall": (
                cluster_mean[activity] - overall_mean[activity]
            ),
            "difference_from_others": (
                cluster_mean[activity] - other_mean[activity]
            )
        })

differences = pd.DataFrame(cluster_differences)

# Xếp hạng hoạt động theo mức chênh lệch với các cụm còn lại
differences = differences.sort_values(
    ["cluster", "difference_from_others"],
    ascending=[True, False]
)

differences.to_csv(
    OUTPUT_DIR / "cluster_distinguishing_activities.csv",
    index=False,
    encoding="utf-8-sig"
)

# In 8 hoạt động có điểm cao tương đối nhất và 5 hoạt động
# thấp tương đối nhất của từng cụm
for cluster_id in sorted(labels.unique()):
    subset = differences[differences["cluster"] == cluster_id]

    print(f"\n===== CLUSTER {cluster_id} =====")

    print("\n8 hoạt động cao hơn các cụm còn lại nhiều nhất:")
    print(
        subset.nlargest(8, "difference_from_others")[
            ["activity", "cluster_mean", "other_clusters_mean",
             "difference_from_others"]
        ].to_string(index=False)
    )

    print("\n5 hoạt động thấp hơn các cụm còn lại nhiều nhất:")
    print(
        subset.nsmallest(5, "difference_from_others")[
            ["activity", "cluster_mean", "other_clusters_mean",
             "difference_from_others"]
        ].to_string(index=False)
    )



===== CLUSTER 0 =====

8 hoạt động cao hơn các cụm còn lại nhiều nhất:
                                            activity  cluster_mean  other_clusters_mean  difference_from_others
                         Handling and Moving Objects      4.172745             3.409731                0.763014
              Performing General Physical Activities      3.293137             2.972603                0.320535
                  Controlling Machines and Processes      2.970196             2.811556                0.158640
      Repairing and Maintaining Mechanical Equipment      2.041716             2.094201               -0.052485
Operating Vehicles, Mechanized Devices, or Equipment      1.824608             1.969901               -0.145293
      Inspecting Equipment, Structures, or Materials      2.819902             3.152673               -0.332771
      Repairing and Maintaining Electronic Equipment      1.521225             1.995757               -0.474531
  Performing for or Working Dire